# 01. Загрузка, EDA и очистка

Проверка пропусков, идентификаторов, групп, возраста и последовательности событий.

Каждый ноутбук запускается независимо: **Restart Kernel → Run All**. Расчёты импортируются из `src/ab_test`. Исходные данные не изменяются.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'ab_test').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from IPython.display import display
pd.set_option('display.max_columns', None)
from ab_test.data import load_data, prepare_data
users, payments = load_data()
users_clean, payments_clean, audit = prepare_data(users, payments)


Изучение структуры

In [2]:
display(users.head())
display(payments.head())

users.info()
payments.info()

,user_id,group,age,city,device_type
0,1001,A,31,Москва,ios
1,1002,A,31,Москва,ios
2,1003,A,25,Москва,android
3,1004,A,20,Москва,ios
4,1005,A,34,Москва,ios


,payment_id,user_id,group,step1_opened,step2_entered,step3_confirmed,step4_success
0,5001,1001.0,A,2024-04-16 06:06:00,2024-04-16 06:10:00,NaT,NaT
1,5002,1002.0,A,2024-04-12 07:38:00,2024-04-12 07:41:00,NaT,NaT
2,5003,1003.0,A,2024-04-26 20:03:00,2024-04-26 20:07:00,NaT,NaT
3,5004,1004.0,A,2024-04-09 16:02:00,NaT,NaT,NaT
4,5005,1004.0,A,2024-04-04 03:46:00,2024-04-04 03:54:00,2024-04-04 03:57:00,2024-04-04 04:02:00


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1504 entries, 0 to 1503
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   user_id      1504 non-null   int64 
 1   group        1503 non-null   object
 2   age          1504 non-null   int64 
 3   city         1503 non-null   object
 4   device_type  1504 non-null   object
dtypes: int64(2), object(3)
memory usage: 58.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2661 entries, 0 to 2660
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   payment_id       2661 non-null   int64         
 1   user_id          2660 non-null   float64       
 2   group            2661 non-null   object        
 3   step1_opened     2661 non-null   datetime64[ns]
 4   step2_entered    2256 non-null   datetime64[ns]
 5   step3_confirmed  1719 non-null   datetime64[ns]
 6   step4_success    1522

Поиск пропусков

In [3]:
print('Пропуски в пользователях:')
print(users.isna().sum())

print('\nПропуски в платежах:')
print(payments.isna().sum())

Пропуски в пользователях:
user_id        0
group          1
age            0
city           1
device_type    0
dtype: int64

Пропуски в платежах:
payment_id            0
user_id               1
group                 0
step1_opened          0
step2_entered       405
step3_confirmed     942
step4_success      1139
dtype: int64


Проверка дубликатов

In [4]:
print(
    'Дубликаты user_id:',
    users['user_id'].duplicated().sum()
)

print(
    'Дубликаты payment_id:',
    payments['payment_id'].duplicated().sum()
)

Дубликаты user_id: 0
Дубликаты payment_id: 0


Проверка распределения групп

In [5]:
print(users['group'].value_counts(dropna=False))

print(payments['group'].value_counts(dropna=False))

group
A      752
B      751
NaN      1
Name: count, dtype: int64
group
B    1352
A    1309
Name: count, dtype: int64


Этап 2. Поиск аномалий и некорректных записей

Найдём пользователей с пропусками

In [6]:
# Пользователи без группы
display(users[users['group'].isna()])

# Пользователи без города
display(users[users['city'].isna()])

,user_id,group,age,city,device_type
1500,9001,NaN,25,Москва,ios


,user_id,group,age,city,device_type
1503,9004,A,30,NaN,android


Проверим аномальные значения возраста

In [7]:
print(users['age'].describe())

# Подозрительные значения
display(
    users[
        (users['age'] < 0) |
        (users['age'] > 110)
    ]
)

count    1504.000000
mean       32.745346
std         8.410915
min        -5.000000
25%        27.000000
50%        33.000000
75%        38.000000
max       150.000000
Name: age, dtype: float64


,user_id,group,age,city,device_type
1501,9002,A,-5,СПб,android
1502,9003,B,150,Регион,ios


Найдём платежи без пользователя

In [8]:
display(
    payments[
        payments['user_id'].isna()
    ]
)

,payment_id,user_id,group,step1_opened,step2_entered,step3_confirmed,step4_success
2660,99003,NaN,A,2024-04-12 09:15:00,NaT,NaT,NaT


Проверим соответствие групп A/B

In [9]:
df_check = payments.merge(
    users[['user_id', 'group']],
    on='user_id',
    how='left',
    suffixes=('_payment', '_user'),
    validate='many_to_one'
)

display(df_check.head())

,payment_id,user_id,group_payment,step1_opened,step2_entered,step3_confirmed,step4_success,group_user
0,5001,1001.0,A,2024-04-16 06:06:00,2024-04-16 06:10:00,NaT,NaT,A
1,5002,1002.0,A,2024-04-12 07:38:00,2024-04-12 07:41:00,NaT,NaT,A
2,5003,1003.0,A,2024-04-26 20:03:00,2024-04-26 20:07:00,NaT,NaT,A
3,5004,1004.0,A,2024-04-09 16:02:00,NaT,NaT,NaT,A
4,5005,1004.0,A,2024-04-04 03:46:00,2024-04-04 03:54:00,2024-04-04 03:57:00,2024-04-04 04:02:00,A


In [10]:
group_errors = df_check[
    (df_check['user_id'].notna()) &
    (df_check['group_payment'] != df_check['group_user'])
]

display(
    group_errors[
        [
            'payment_id',
            'user_id',
            'group_payment',
            'group_user'
        ]
    ]
)

,payment_id,user_id,group_payment,group_user
2659,99002,1010.0,B,A


## Правила очистки
Оставляем пользователей A/B; присоединяем только их платежи. Группа берётся из таблицы пользователей. Аномалии возраста и пропуски города сохраняются. Дубликаты ID и нарушения воронки останавливают расчёт.

In [11]:
display(pd.Series(audit))
display(users_clean.head())
display(payments_clean.head())

raw_users                                                                1504
raw_payments                                                             2661
excluded_users                                                              1
payments_missing_user_id                                                    1
payments_unknown_user                                                       0
group_mismatches                                                            1
abnormal_age                                                                2
missing_city                                                                1
duplicate_users                                                             0
duplicate_payments                                                          0
clean_users                                                              1503
clean_payments                                                           2660
excluded_payments                                               

,user_id,group,age,city,device_type
0,1001,A,31,Москва,ios
1,1002,A,31,Москва,ios
2,1003,A,25,Москва,android
3,1004,A,20,Москва,ios
4,1005,A,34,Москва,ios


,payment_id,user_id,group,step1_opened,step2_entered,step3_confirmed,step4_success,age,city,device_type,success
0,5001,1001.0,A,2024-04-16 06:06:00,2024-04-16 06:10:00,NaT,NaT,31,Москва,ios,0
1,5002,1002.0,A,2024-04-12 07:38:00,2024-04-12 07:41:00,NaT,NaT,31,Москва,ios,0
2,5003,1003.0,A,2024-04-26 20:03:00,2024-04-26 20:07:00,NaT,NaT,25,Москва,android,0
3,5004,1004.0,A,2024-04-09 16:02:00,NaT,NaT,NaT,20,Москва,ios,0
4,5005,1004.0,A,2024-04-04 03:46:00,2024-04-04 03:54:00,2024-04-04 03:57:00,2024-04-04 04:02:00,20,Москва,ios,1


## Сохранение подготовленных данных
CSV нужны для проверки и внешнего использования. Остальные ноутбуки повторяют детерминированную очистку из raw, поэтому не зависят от состояния другого ядра.

In [12]:
processed = ROOT / 'data' / 'processed'
processed.mkdir(parents=True, exist_ok=True)
users_clean.to_csv(processed / 'users_clean.csv', index=False, encoding='utf-8-sig')
payments_clean.to_csv(processed / 'payments_clean.csv', index=False, encoding='utf-8-sig')